# F5-TTS Voice Cloning API Server for Kaggle (Free GPU Edition)

This notebook starts a high-performance **F5-TTS Voice Cloning** API server on Kaggle's free GPU instance (T4 GPU). It uses **Pinggy** to securely tunnel the API endpoint to a public URL that you can connect to from the Dubber PRO dashboard.

### How to run:
1. Open this notebook on Kaggle.
2. Enable **GPU T4 x2** or **GPU T4** under Notebook Settings.
3. Turn on the **Internet** toggle in Notebook Settings.
4. Click **Run All**.
5. Copy the printed **F5-TTS PUBLIC URL** and paste it into your local Dubber PRO dashboard!

## Step 1: Install Dependencies
We install F5-TTS along with FastAPI, Uvicorn, and `nest-asyncio` to host the programmatic HTTP synthesis endpoint.

In [ ]:
# Install packages
!pip install -q f5-tts fastapi uvicorn python-multipart nest-asyncio

## Step 2: Write FastAPI Server Script
We will write the server script `f5_server.py`. It exposes a `/synthesize` endpoint which receives the speaker's vocal reference, parses the transcription context, and runs `f5-tts_infer-cli` programmatically.

In [ ]:
%%writefile f5_server.py
import os
import tempfile
import subprocess
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse

app = FastAPI(title="Kaggle F5-TTS Synthesis Server")

@app.get("/")
def index():
    return {"status": "online", "model": "F5-TTS"}

@app.post("/synthesize")
async def synthesize(
    ref_audio: UploadFile = File(...),
    ref_text: str = Form(...),
    gen_text: str = Form(...)
):
    # Create a temporary environment to perform the synthesis run
    with tempfile.TemporaryDirectory() as tmpdir:
        ref_path = os.path.join(tmpdir, "ref.wav")
        out_path = os.path.join(tmpdir, "out.wav")
        
        # Write reference audio clip to disk
        with open(ref_path, "wb") as f:
            f.write(await ref_audio.read())
            
        # Prepare CLI command arguments for inference
        cmd = [
            "f5-tts_infer-cli",
            "--model", "F5-TTS",
            "--ref_audio", ref_path,
            "--ref_text", ref_text,
            "--gen_text", gen_text,
            "--output_file", out_path
        ]
        
        print(f"Executing: {' '.join(cmd)}")
        res = subprocess.run(cmd, capture_output=True, text=True)
        
        # Fallback to output directory if --output_file isn't supported
        if res.returncode != 0:
            print(f"Warning: Standard output_file command failed: {res.stderr}. Trying output_dir fallback...")
            cmd_fallback = [
                "f5-tts_infer-cli",
                "--model", "F5-TTS",
                "--ref_audio", ref_path,
                "--ref_text", ref_text,
                "--gen_text", gen_text,
                "--output_dir", tmpdir
            ]
            res_fallback = subprocess.run(cmd_fallback, capture_output=True, text=True)
            
            generated_files = [f for f in os.listdir(tmpdir) if f.endswith(".wav") and f != "ref.wav"]
            if generated_files:
                out_path = os.path.join(tmpdir, generated_files[0])
            else:
                return {
                    "error": "Inference failed",
                    "stdout": res.stdout + "\n" + res_fallback.stdout,
                    "stderr": res.stderr + "\n" + res_fallback.stderr
                }
        
        # Read bytes before returning so temporary folder can clean up safely
        temp_out = os.path.join(os.getcwd(), "temp_synthesized.wav")
        if os.path.exists(out_path):
            import shutil
            shutil.copy(out_path, temp_out)
            return FileResponse(temp_out, media_type="audio/wav", filename="synthesized.wav")
        else:
            return {"error": "Synthesized output file not generated.", "stderr": res.stderr}

## Step 3: Run SSH Tunnel & Start Uvicorn Server
We start Pinggy to set up the SSH tunnel in a background thread, parse the public URL, and start the FastAPI server on port 8000.

In [ ]:
import subprocess
import threading
import re
import time
import nest_asyncio
import uvicorn

# Allow nested asyncio loops in notebooks
nest_asyncio.apply()

def start_ssh_tunnel():
    print("[Tunnel] Initializing SSH connection to Pinggy...")
    # Connect via SSH to reverse-tunnel port 8000
    cmd = "ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -p 443 -R0:localhost:8000 qr@pinggy.io"
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    # Track and print the public tunnel address
    for line in iter(process.stdout.readline, ''):
        cleaned_line = line.strip()
        if "http" in cleaned_line:
            urls = re.findall(r'https?://[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?::\d+)?', cleaned_line)
            for url in urls:
                print("\n" + "*"*70)
                print(f" 👉 👉 👉 YOUR F5-TTS PUBLIC URL: {url} 👈 👈 👈 ")
                print("*"*70 + "\n")
        else:
            print(f"[Tunnel] {cleaned_line}")

# Start tunnel in the background
tunnel_thread = threading.Thread(target=start_ssh_tunnel, daemon=True)
tunnel_thread.start()

# Wait 3 seconds for tunnel to register
time.sleep(3)

# Start FastAPI app server
print("[Server] Starting FastAPI Uvicorn engine on port 8000...")
uvicorn.run("f5_server:app", host="127.0.0.1", port=8000, log_level="info")